In [4]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 38.3 MB/s eta 0:00:00


In [5]:
from ultralytics import YOLO
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import os

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [7]:
from zipfile import ZipFile

with ZipFile("/content/food_img_data_resized (1).zip","r") as zip_ref:
    zip_ref.extractall("/content")

In [8]:
model = YOLO("yolov8n-cls.pt")

In [9]:
model.train(
    data="/content/food_img_data_resized",   # Dataset folder
    epochs=30,
    imgsz=224,
    batch=32,
    device=0
)

Ultralytics 8.4.98 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/food_img_data_resized, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_m

ultralytics.utils.metrics.ClassifyMetrics object with attributes:

confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x78188c6364b0>
curves: []
curves_results: []
fitness: 0.9806547462940216
keys: ['metrics/accuracy_top1', 'metrics/accuracy_top5']
results_dict: {'metrics/accuracy_top1': 0.9702380895614624, 'metrics/accuracy_top5': 0.9910714030265808, 'fitness': 0.9806547462940216}
save_dir: PosixPath('/content/runs/classify/train')
speed: {'preprocess': 0.1418464642854577, 'inference': 0.7050351160716992, 'loss': 0.00020338988059891215, 'postprocess': 0.000295389881105599}
top1: 0.9702380895614624
top5: 0.9910714030265808

In [10]:
import glob

best_model = glob.glob("/content/runs/classify/**/weights/best.pt", recursive=True)

print(best_model)

['/content/runs/classify/train/weights/best.pt']


In [11]:
model = YOLO(best_model[0])

In [22]:
nutrition = pd.read_excel("/content/Nutrition_Table_15_Food_Classes.xlsx")

nutrition.head()

,Food Name,Serving Size (g),Calories (kcal),Protein (g),Carbohydrates (g),Fat (g),Fiber (g),Sugar (g),Sodium (mg)
0,Biryani,350,720,32,78,28,3,3,1200
1,Pizza,250,620,22,70,24,4,6,950
2,Vegetable Salad,200,180,5,18,8,6,7,220
3,Fruits Salad,250,150,2,38,1,5,30,10
4,Burger,250,580,26,45,30,4,8,950


In [24]:
from google.colab import files

uploaded = files.upload()


Saving food_img_data_resized (1).zip to food_img_data_resized (1) (3).zip


To ensure the `model` is defined, run the following cell. This cell loads the best-trained weights into a `YOLO` model instance.

In [26]:
model = YOLO(best_model[0])

In [ ]:
!pip install -q ultralytics gradio openpyxl

import gradio as gr
import pandas as pd
from ultralytics import YOLO

# Load Model
model = YOLO("/content/runs/classify/train/weights/best.pt")

# Load Nutrition Table
nutrition = pd.read_excel("/content/Nutrition_Table_15_Food_Classes.xlsx")

# Check column names
print("Columns in Excel:")
print(nutrition.columns.tolist())

# Change this if your first column has a different name
food_column = nutrition.columns[0]

def predict_food(image):
    try:
        # Predict
        results = model.predict(image, verbose=False)

        predicted = results[0].names[results[0].probs.top1]
        confidence = float(results[0].probs.top1conf) * 100

        print("Prediction:", predicted)

        # Find nutrition row
        food = nutrition[
            nutrition[food_column].astype(str).str.lower() == predicted.lower()
        ]

        if not food.empty:

            row = food.iloc[0]

            nutrition_text = ""

            for col in nutrition.columns:
                nutrition_text += f"{col}: {row[col]}\n"

        else:

            nutrition_text = "Nutrition information not found."

        return (
            predicted,
            f"{confidence:.2f} %",
            nutrition_text
        )

    except Exception as e:
        print(e)
        return (
            "Error",
            "Error",
            str(e)
        )

demo = gr.Interface(
    fn=predict_food,
    inputs=gr.Image(type="pil", label="Upload Image"),
    outputs=[
        gr.Textbox(label="Predicted Food"),
        gr.Textbox(label="Confidence"),
        gr.Textbox(label="Nutrition Information")
    ],
    title="Food Detection using YOLO",
    description="Upload a food image and click Submit."
)

demo.launch(debug=True)

Columns in Excel:
['Food Name', 'Serving Size (g)', 'Calories (kcal)', 'Protein (g)', 'Carbohydrates (g)', 'Fat (g)', 'Fiber (g)', 'Sugar (g)', 'Sodium (mg)']
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://e5b9cccb599f441c8a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Prediction: noodles
Prediction: Pavbhaji
Prediction: pizza
Prediction: pasta
Prediction: sandwiches
Prediction: sushi
Prediction: dosa
